In [ ]:
# 本 Notebook 用于加载和检查原始数据，包括结构、类型、缺失值、重复值、异常值和其他数据质量问题。


# 1. Load the Raw Dataset

## Purpose

This section loads the original Rossmann Store Sales data into Python for subsequent data inspection.

The project uses the following two primary input files:

- `train.csv`: historical daily store sales data, including sales, customer counts, promotions, holidays, and store identifiers.
- `store.csv`: supplementary store-level information, such as store type, assortment type, competition information, and long-term promotion information.

The files are read directly from the raw data directory and are not modified during this step.

## Input

- `data/raw/train.csv`
- `data/raw/store.csv`

## Output

This step does not generate any new files.

Instead, two Pandas DataFrames are created in memory:

- `train`: contains the historical daily sales records.
- `store`: contains the supplementary store information.

These DataFrames will be used in the following sections for structural inspection and data quality checking.

## Method

Python's `pathlib.Path` is used to construct file paths, while `pandas.read_csv()` is used to read the CSV files.

`Path` is preferred to hard-coded absolute paths because relative project paths make the notebook more portable and reproducible across different computers.

`pandas.read_csv()` is used because the dataset is stored in structured CSV format and Pandas provides direct support for tabular data loading, inspection, cleaning, statistical analysis, and later preprocessing.

Using Python's low-level `csv` module would also be possible, but it would require additional manual processing to obtain column-based operations, missing-value handling, descriptive statistics, and other functionality that Pandas provides directly.

## Principle

A CSV file stores tabular data as text, where each row represents a record and delimiters separate the fields.

`pandas.read_csv()` parses the CSV file and converts it into a DataFrame, a two-dimensional labelled data structure in which:

- rows represent observations;
- columns represent variables;
- each column is assigned an inferred data type.

At this stage, Pandas initially infers the data types from the file contents. These inferred types will be checked later rather than being manually specified during loading, because incorrect or unexpected data types may themselves reveal data quality issues.

## Evaluation Criteria

The loading step is considered successful if:

1. both input files are found;
2. both files can be parsed without errors;
3. both resulting DataFrames are non-empty;
4. the loaded objects are valid Pandas DataFrames.

Detailed checks of row counts, columns, data types, missing values, duplicate records, invalid values, and other data quality issues will be performed in subsequent sections.

In [1]:
# Load the original Rossmann datasets without modifying the raw files.

from pathlib import Path
import pandas as pd


# The notebook is located in the notebooks/ directory.
# Therefore, its parent directory is treated as the project root.
PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
STORE_PATH = PROJECT_ROOT / "data" / "raw" / "store.csv"


# Check whether the required input files exist before loading them.
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"train.csv was not found: {TRAIN_PATH}")

if not STORE_PATH.exists():
    raise FileNotFoundError(f"store.csv was not found: {STORE_PATH}")


# Load the raw CSV files into Pandas DataFrames.
train = pd.read_csv(TRAIN_PATH)
store = pd.read_csv(STORE_PATH)


# Basic loading validation.
if train.empty:
    raise ValueError("train.csv was loaded, but the resulting DataFrame is empty.")

if store.empty:
    raise ValueError("store.csv was loaded, but the resulting DataFrame is empty.")


print("Raw datasets loaded successfully.")
print(f"train.csv: {TRAIN_PATH}")
print(f"store.csv: {STORE_PATH}")
print()
print(f"train DataFrame: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"store DataFrame: {store.shape[0]:,} rows × {store.shape[1]} columns")

Raw datasets loaded successfully.
train.csv: d:\Coding\NTUHomeworks\AssignmentOfCA6000\data\raw\train.csv
store.csv: d:\Coding\NTUHomeworks\AssignmentOfCA6000\data\raw\store.csv

train DataFrame: 1,017,209 rows × 9 columns
store DataFrame: 1,115 rows × 10 columns


C:\Users\31729\AppData\Local\Temp\ipykernel_11484\3578457680.py:24: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)


# 2. Investigate the `StateHoliday` Data Type Warning

## Observation

The raw datasets were loaded successfully. However, Pandas produced a `DtypeWarning` while reading `train.csv`, indicating that the `StateHoliday` column contains mixed data types.

The dataset is not modified at this stage. The warning is investigated first so that the original representation of the data can be understood before any cleaning or type conversion is performed.

## Input

The analysis uses the `train` DataFrame already loaded from:

- `data/raw/train.csv`

No additional input file is read in this step.

## Output

This step does not create or modify any file.

The code produces an inspection table in the notebook showing:

- each distinct representation found in `StateHoliday`;
- the underlying Python type of each representation;
- the number of records using each representation.

The result is used to determine whether the warning is caused by genuinely different categories or by inconsistent representations of the same category.

## Method

The `StateHoliday` column is first inspected using its Pandas data type. Its distinct values are then identified with `drop_duplicates()`.

For every distinct value:

1. `repr()` is used to display the exact representation of the value;
2. `type()` is used to identify its underlying Python type;
3. Boolean comparison is used to count how many records contain that exact value.

`repr()` is used instead of ordinary string display because values such as the integer `0` and the string `"0"` can look similar when printed normally even though they are different Python objects.

The column is not converted to `str`, `category`, or another type during this inspection because doing so would alter the original representation and could hide the source of the warning.

Similarly, the warning is not suppressed by changing the CSV loading options at this stage. The purpose of data inspection is to identify the original condition of the dataset before deciding how it should be cleaned.

## Principle

A Pandas column with the `object` data type can contain different kinds of Python objects within the same column.

For example, two values may appear visually similar but have different representations:

- `0` is an integer;
- `"0"` is a string.

Although both may represent the same conceptual category, they are not the same Python value.

When Pandas reads a large CSV file, it may infer data types from different portions of the file. If different portions suggest different types for the same column, Pandas may generate a `DtypeWarning` and store the resulting column using the general `object` data type.

Therefore, the warning is treated as a data representation issue that must be inspected before cleaning.

## Evaluation Criteria

The result should be evaluated by checking:

1. how many distinct representations are present;
2. whether multiple Python types occur in the same column;
3. whether visually similar values are stored using different types;
4. whether any unexpected or missing categories are present.

If the same conceptual category is represented by different Python types, this will be recorded as a data consistency issue and handled later during data cleaning.

No correction is performed in this section.

In [2]:
# Inspect the exact values and Python types stored in StateHoliday.

print("Pandas dtype:")
print(train["StateHoliday"].dtype)

print("\nNumber of distinct representations:")
print(train["StateHoliday"].nunique(dropna=False))


stateholiday_profile = []

for value in train["StateHoliday"].drop_duplicates():

    if pd.isna(value):
        count = train["StateHoliday"].isna().sum()
    else:
        count = (train["StateHoliday"] == value).sum()

    stateholiday_profile.append(
        {
            "Value (repr)": repr(value),
            "Python type": type(value).__name__,
            "Count": int(count)
        }
    )


stateholiday_profile = (
    pd.DataFrame(stateholiday_profile)
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

display(stateholiday_profile)

Pandas dtype:
object

Number of distinct representations:
5


,Value (repr),Python type,Count
0,'0',str,855087
1,0,int,131072
2,'a',str,20260
3,'b',str,6690
4,'c',str,4100


# 3. Inspect the Dataset Structure

## Purpose

After confirming that the raw files can be loaded, this section examines the basic structure of both datasets.

The purpose is to understand how the observations and variables are represented before performing detailed data quality checks.

The previous inspection also identified an inconsistent representation in `StateHoliday`: the value `0` appears as both a string (`"0"`) and an integer (`0`). This issue is recorded for later cleaning and is not corrected in this notebook.

## Input

This section uses the two DataFrames already loaded into memory:

- `train`, originating from `data/raw/train.csv`
- `store`, originating from `data/raw/store.csv`

No additional files are read.

## Output

No new files are created.

The code displays:

1. the first five records of each dataset;
2. the number of rows and columns;
3. a schema summary for every column, including:
   - column name;
   - Pandas data type;
   - number of non-null values;
   - number of null values;
   - percentage of null values.

These outputs provide an initial structural overview and help identify columns that require more detailed inspection.

## Method

`DataFrame.head()` is used to display the first five observations because it provides a compact preview of the dataset without printing all records.

`DataFrame.shape` is used to obtain the number of rows and columns.

Column-level structural information is obtained using:

- `dtypes` for inferred Pandas data types;
- `notna().sum()` for non-null counts;
- `isna().sum()` for missing-value counts.

The missing-value percentage is calculated as:

`number of missing values / total number of rows × 100`

A structured summary table is used instead of relying only on `DataFrame.info()` because the table is easier to compare across columns and explicitly includes the missing-value percentage.

`DataFrame.info()` is useful for quick interactive inspection, but its text-based output is less convenient for later interpretation and comparison.

## Principle

A tabular dataset can be described at two main structural levels:

- **row level**: each row represents an observation;
- **column level**: each column represents a variable.

Before analysing the values themselves, it is necessary to confirm:

- whether the expected variables are present;
- whether the dataset size is plausible;
- how Pandas has interpreted each variable's type;
- whether any columns contain missing observations.

Data types are particularly important because later statistical operations and preprocessing methods depend on whether a variable is treated as numeric, categorical, textual, or temporal.

Missing-value counts are included at this stage as a structural indicator. Detailed decisions about whether and how missing values should be handled will be made later.

## Evaluation Criteria

The result should be evaluated by checking:

1. whether the first records appear correctly aligned with their column names;
2. whether both datasets contain a plausible number of rows and columns;
3. whether each inferred data type is reasonable for the meaning of the variable;
4. which columns contain missing values;
5. whether the proportion of missing values is small or substantial;
6. whether any columns require further investigation before cleaning.

No data is modified in this section.

In [3]:
# Inspect the basic structure, data types, and missing-value status of both datasets.

def build_schema_summary(df):
    """Create a structural summary for each column in a DataFrame."""
    
    summary = pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.astype(str).values,
        "Non-Null": df.notna().sum().values,
        "Missing": df.isna().sum().values,
        "Missing (%)": (
            df.isna().mean().values * 100
        )
    })

    summary["Missing (%)"] = summary["Missing (%)"].round(2)

    return summary


print("TRAIN DATASET")
print(f"Shape: {train.shape[0]:,} rows × {train.shape[1]} columns")
display(train.head())

print("Schema summary:")
display(build_schema_summary(train))


print("\nSTORE DATASET")
print(f"Shape: {store.shape[0]:,} rows × {store.shape[1]} columns")
display(store.head())

print("Schema summary:")
display(build_schema_summary(store))

TRAIN DATASET
Shape: 1,017,209 rows × 9 columns


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


Schema summary:


,Column,Dtype,Non-Null,Missing,Missing (%)
0,Store,int64,1017209,0,0.0
1,DayOfWeek,int64,1017209,0,0.0
2,Date,str,1017209,0,0.0
3,Sales,int64,1017209,0,0.0
4,Customers,int64,1017209,0,0.0
5,Open,int64,1017209,0,0.0
6,Promo,int64,1017209,0,0.0
7,StateHoliday,object,1017209,0,0.0
8,SchoolHoliday,int64,1017209,0,0.0



STORE DATASET
Shape: 1,115 rows × 10 columns


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


Schema summary:


,Column,Dtype,Non-Null,Missing,Missing (%)
0,Store,int64,1115,0,0.00
1,StoreType,str,1115,0,0.00
2,Assortment,str,1115,0,0.00
3,CompetitionDistance,float64,1112,3,0.27
4,CompetitionOpenSinceMonth,float64,761,354,31.75
5,CompetitionOpenSinceYear,float64,761,354,31.75
6,Promo2,int64,1115,0,0.00
7,Promo2SinceWeek,float64,571,544,48.79
8,Promo2SinceYear,float64,571,544,48.79
9,PromoInterval,str,571,544,48.79


# 4. Investigate Missing Values in the Store Dataset

## Purpose

The structural inspection showed that `train` contains no missing values, while several columns in `store` contain missing observations.

However, a missing value does not necessarily represent a data error. Some variables are only applicable under specific business conditions.

For example, the fields describing the start and interval of `Promo2` may only be applicable to stores that participate in `Promo2`.

Therefore, this section investigates whether the observed missing values are random data losses or structurally related to other variables before any cleaning decision is made.

## Input

This section uses the `store` DataFrame already loaded from:

- `data/raw/store.csv`

No additional input file is read.

## Output

No new file is created or modified.

The code produces three inspection results:

1. a summary of columns containing missing values;
2. a comparison of `Promo2` status with missing values in its associated detail fields;
3. a consistency check of missing values in the competition-related fields.

These results will be used later to decide whether a missing value should be retained, encoded, imputed, or otherwise handled during data cleaning.

## Method

Missing values are detected using `isna()`.

For the `Promo2` fields, records are grouped by the `Promo2` indicator. Within each group, the code counts whether:

- all associated `Promo2` detail fields are missing;
- at least one detail field is missing;
- all detail fields are present.

This conditional comparison is preferred to immediately applying statistical imputation because the missing values may represent "not applicable" rather than unknown numerical information.

For the competition-related fields, Boolean missing-value masks are compared to determine whether `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` are missing together or independently.

This approach is used instead of immediately replacing missing values with a mean, median, mode, or constant because such replacement should only be performed after the semantic reason for missingness is understood.

## Principle

Missing data can have different meanings.

A missing value may represent:

- unavailable or unrecorded information;
- an unknown value;
- a variable that is not applicable to a particular observation;
- a value lost during data collection or processing.

These situations should not automatically receive the same treatment.

In particular, if a store does not participate in `Promo2`, then variables such as `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval` may naturally have no applicable value. Such missingness is structurally determined by another variable and should be distinguished from accidental data loss.

Similarly, missing competition opening dates should be examined separately from missing competition distance because these variables describe different aspects of the competition information.

## Evaluation Criteria

The results should be evaluated by checking:

1. whether the `Promo2` detail fields are systematically missing when `Promo2 = 0`;
2. whether those fields are present when `Promo2 = 1`;
3. whether the three `Promo2` detail fields are missing together or inconsistently;
4. whether `CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` are missing together;
5. whether competition opening dates can be missing even when `CompetitionDistance` is available.

If missing values are fully explained by another variable, they should be treated as structural missingness rather than automatically classified as data errors.

No missing values are modified in this section.

In [4]:
# Investigate whether missing values in store.csv are structural or unexpected.

# ------------------------------------------------------------
# 1. Summary of columns containing missing values
# ------------------------------------------------------------

store_missing = build_schema_summary(store)

store_missing = store_missing.loc[
    store_missing["Missing"] > 0,
    ["Column", "Missing", "Missing (%)"]
].reset_index(drop=True)

print("Columns containing missing values:")
display(store_missing)


# ------------------------------------------------------------
# 2. Investigate Promo2-related missing values
# ------------------------------------------------------------

promo2_detail_cols = [
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval"
]

promo2_rows = []

for promo2_value, group in store.groupby("Promo2"):

    all_missing = group[promo2_detail_cols].isna().all(axis=1).sum()
    any_missing = group[promo2_detail_cols].isna().any(axis=1).sum()
    all_present = group[promo2_detail_cols].notna().all(axis=1).sum()

    promo2_rows.append({
        "Promo2": promo2_value,
        "Stores": len(group),
        "All detail fields missing": int(all_missing),
        "Any detail field missing": int(any_missing),
        "All detail fields present": int(all_present)
    })

promo2_check = pd.DataFrame(promo2_rows)

print("\nPromo2 missing-value consistency:")
display(promo2_check)


# ------------------------------------------------------------
# 3. Investigate competition-related missing values
# ------------------------------------------------------------

distance_missing = store["CompetitionDistance"].isna()
month_missing = store["CompetitionOpenSinceMonth"].isna()
year_missing = store["CompetitionOpenSinceYear"].isna()

competition_check = pd.DataFrame({
    "Condition": [
        "CompetitionDistance missing",
        "CompetitionOpenSinceMonth missing",
        "CompetitionOpenSinceYear missing",
        "Month and year both missing",
        "Only month missing",
        "Only year missing",
        "Distance available but month and year both missing"
    ],
    "Count": [
        int(distance_missing.sum()),
        int(month_missing.sum()),
        int(year_missing.sum()),
        int((month_missing & year_missing).sum()),
        int((month_missing & ~year_missing).sum()),
        int((~month_missing & year_missing).sum()),
        int((~distance_missing & month_missing & year_missing).sum())
    ]
})

print("\nCompetition-related missing-value consistency:")
display(competition_check)

Columns containing missing values:


,Column,Missing,Missing (%)
0,CompetitionDistance,3,0.27
1,CompetitionOpenSinceMonth,354,31.75
2,CompetitionOpenSinceYear,354,31.75
3,Promo2SinceWeek,544,48.79
4,Promo2SinceYear,544,48.79
5,PromoInterval,544,48.79



Promo2 missing-value consistency:


,Promo2,Stores,All detail fields missing,Any detail field missing,All detail fields present
0,0,544,544,544,0
1,1,571,0,0,571



Competition-related missing-value consistency:


,Condition,Count
0,CompetitionDistance missing,3
1,CompetitionOpenSinceMonth missing,354
2,CompetitionOpenSinceYear missing,354
3,Month and year both missing,354
4,Only month missing,0
5,Only year missing,0
6,Distance available but month and year both mis...,351


# 5. Check Duplicate Records and Key Uniqueness

## Purpose

This section checks whether either dataset contains duplicated observations.

Two types of duplication are examined:

1. **full-row duplicates**, where all column values are identical;
2. **business-key duplicates**, where multiple rows represent the same logical observation even if other column values differ.

For the daily sales dataset, one store should normally have at most one record for a given date. Therefore, the combination of `Store` and `Date` is treated as the expected business key.

For the store information dataset, each store should have one store-level record. Therefore, `Store` is treated as the expected business key.

No duplicate records are removed in this section.

## Input

This section uses the DataFrames already loaded into memory:

- `train`
- `store`

No additional files are read.

## Output

No file is created or modified.

The code produces a summary table containing:

- the number of full-row duplicate records;
- the number of rows involved in duplicated business keys;
- the number of distinct duplicated business keys.

If any duplicated business keys are detected, example records are also displayed for further inspection.

## Method

`DataFrame.duplicated()` is used to detect completely identical rows.

Business-key duplication is checked using `duplicated(subset=..., keep=False)`:

- `["Store", "Date"]` is used for `train`;
- `["Store"]` is used for `store`.

`keep=False` marks every row involved in a duplicated key rather than marking only the later occurrence.

The number of distinct duplicated keys is also calculated separately because the number of affected rows and the number of duplicated logical observations are not necessarily the same.

This method is preferred to immediately using `drop_duplicates()` because duplicate detection and duplicate removal are different tasks. Removing rows before understanding whether they are exact duplicates or conflicting records could result in valid information being discarded.

## Principle

A full-row duplicate occurs when two or more rows contain exactly the same values in every column.

A business-key duplicate is more important from a data-integrity perspective. It occurs when multiple records share the identifier that should uniquely define an observation.

For example, if two rows have the same `Store` and `Date` but different `Sales` values, they are not full-row duplicates, but they still represent conflicting records for the same store-day observation.

Therefore, checking only complete-row duplication is insufficient.

The expected uniqueness constraints in this project are:

- `train`: one observation per `Store` and `Date`;
- `store`: one observation per `Store`.

## Evaluation Criteria

The preferred result is:

- zero full-row duplicates in both datasets;
- zero duplicated `Store`-`Date` keys in `train`;
- zero duplicated `Store` keys in `store`.

If full-row duplicates are found, they may represent redundant records.

If duplicated business keys are found but the corresponding rows contain different values, they represent a more serious consistency problem and must be investigated before cleaning.

No duplicate records are removed in this section.

In [5]:
# Check exact duplicate rows and expected business-key uniqueness.

# ------------------------------------------------------------
# 1. Full-row duplicates
# ------------------------------------------------------------

train_full_duplicates = int(train.duplicated().sum())
store_full_duplicates = int(store.duplicated().sum())


# ------------------------------------------------------------
# 2. Business-key duplicates
# ------------------------------------------------------------

# One store should have at most one daily record for each date.
train_key_mask = train.duplicated(
    subset=["Store", "Date"],
    keep=False
)

# Each store should have one store-level information record.
store_key_mask = store.duplicated(
    subset=["Store"],
    keep=False
)


train_key_rows = int(train_key_mask.sum())
store_key_rows = int(store_key_mask.sum())


# Count the number of distinct duplicated keys.
train_duplicate_keys = int(
    train.loc[train_key_mask, ["Store", "Date"]]
    .drop_duplicates()
    .shape[0]
)

store_duplicate_keys = int(
    store.loc[store_key_mask, ["Store"]]
    .drop_duplicates()
    .shape[0]
)


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

duplicate_summary = pd.DataFrame({
    "Dataset": ["train", "store"],
    "Full-row duplicates": [
        train_full_duplicates,
        store_full_duplicates
    ],
    "Rows in duplicated keys": [
        train_key_rows,
        store_key_rows
    ],
    "Distinct duplicated keys": [
        train_duplicate_keys,
        store_duplicate_keys
    ]
})

display(duplicate_summary)


# ------------------------------------------------------------
# 4. Display examples only if duplicated keys exist
# ------------------------------------------------------------

if train_key_rows > 0:
    print("\nExamples of duplicated Store-Date keys in train:")
    display(
        train.loc[train_key_mask]
        .sort_values(["Store", "Date"])
        .head(20)
    )

if store_key_rows > 0:
    print("\nExamples of duplicated Store keys in store:")
    display(
        store.loc[store_key_mask]
        .sort_values("Store")
        .head(20)
    )

,Dataset,Full-row duplicates,Rows in duplicated keys,Distinct duplicated keys
0,train,0,0,0
1,store,0,0,0


# 6. Validate Categorical and Binary Values

## Purpose

This section checks whether categorical and binary variables contain only values that are valid according to their intended meaning.

The previous duplicate check found no full-row duplicates and no duplicated business keys in either dataset. Therefore, no duplicate-related cleaning is currently required.

The current step focuses on domain validity: even when a value is not missing and has a valid data type, it may still be invalid if it falls outside the allowed set of values.

## Input

This section uses the DataFrames already loaded into memory:

- `train`
- `store`

No additional files are read.

## Output

No file is created or modified.

Two validation tables are displayed:

- one for categorical and binary fields in `train`;
- one for categorical and binary fields in `store`.

For each checked field, the output shows:

- the expected set of values;
- the values actually observed in the dataset;
- the number of rows containing invalid values.

## Method

The validation is based on predefined allowed-value sets.

For each field:

1. all observed non-missing values are identified;
2. the observed values are compared with the expected allowed values;
3. `isin()` is used to create a Boolean mask identifying values outside the permitted set;
4. the number of invalid rows is counted.

For `StateHoliday`, a temporary string representation is used only for the validity comparison because the previous inspection showed that the value representing "no state holiday" is stored as both integer `0` and string `"0"`.

The original `StateHoliday` column is not modified. Converting only the temporary validation Series to string allows the semantic categories to be checked without hiding the previously identified type-consistency issue.

This explicit domain-validation approach is preferred to relying only on `unique()` because `unique()` shows what values exist but does not determine whether those values are valid.

## Principle

Data validity and data type are different concepts.

For example, an integer value of `9` would have a valid numeric data type in `DayOfWeek`, but it would still be invalid because the permitted range is 1 to 7.

Similarly, a binary field can be stored correctly as an integer while still containing an invalid value such as `2`.

Categorical validation therefore compares each observed value against a predefined domain of acceptable values.

For a variable \(X\) with an allowed set \(A\), a value is considered valid when:

`X ∈ A`

and invalid when:

`X ∉ A`

This type of rule-based validation is particularly suitable for binary and categorical variables because their valid domains are discrete and known in advance.

## Evaluation Criteria

The preferred result is an invalid-row count of zero for every checked variable.

The expected domains used in this section are:

### `train`

- `DayOfWeek`: 1, 2, 3, 4, 5, 6, 7
- `Open`: 0, 1
- `Promo`: 0, 1
- `StateHoliday`: 0, a, b, c
- `SchoolHoliday`: 0, 1

### `store`

- `StoreType`: a, b, c, d
- `Assortment`: a, b, c
- `Promo2`: 0, 1

If an invalid value is detected, the corresponding rows must be investigated before any correction is performed.

An invalid-row count of zero indicates that the observed categories satisfy the defined domain rules, but it does not by itself guarantee that the entire dataset is error-free.

No values are modified in this section.

In [6]:
# Validate categorical and binary fields against their expected value domains.

def validate_allowed_values(df, column, allowed_values, convert_to_string=False):
    """
    Compare observed values in one column with a predefined set of valid values.
    """

    original = df[column]

    # Use a temporary representation for validation only.
    if convert_to_string:
        checked = original.astype(str)
    else:
        checked = original

    # Missing values are excluded here because they have already been
    # investigated separately.
    invalid_mask = (
        original.notna()
        & ~checked.isin(allowed_values)
    )

    observed_values = (
        checked[original.notna()]
        .drop_duplicates()
        .tolist()
    )

    observed_values = sorted(
        observed_values,
        key=lambda value: str(value)
    )

    expected_values = sorted(
        allowed_values,
        key=lambda value: str(value)
    )

    return {
        "Column": column,
        "Expected values": ", ".join(map(str, expected_values)),
        "Observed values": ", ".join(map(str, observed_values)),
        "Invalid rows": int(invalid_mask.sum())
    }


# ------------------------------------------------------------
# 1. Validate train.csv fields
# ------------------------------------------------------------

train_rules = [
    ("DayOfWeek", {1, 2, 3, 4, 5, 6, 7}, False),
    ("Open", {0, 1}, False),
    ("Promo", {0, 1}, False),
    ("StateHoliday", {"0", "a", "b", "c"}, True),
    ("SchoolHoliday", {0, 1}, False)
]

train_validation = pd.DataFrame([
    validate_allowed_values(
        train,
        column,
        allowed_values,
        convert_to_string
    )
    for column, allowed_values, convert_to_string in train_rules
])

print("TRAIN categorical-value validation:")
display(train_validation)


# ------------------------------------------------------------
# 2. Validate store.csv fields
# ------------------------------------------------------------

store_rules = [
    ("StoreType", {"a", "b", "c", "d"}, False),
    ("Assortment", {"a", "b", "c"}, False),
    ("Promo2", {0, 1}, False)
]

store_validation = pd.DataFrame([
    validate_allowed_values(
        store,
        column,
        allowed_values,
        convert_to_string
    )
    for column, allowed_values, convert_to_string in store_rules
])

print("\nSTORE categorical-value validation:")
display(store_validation)

TRAIN categorical-value validation:


,Column,Expected values,Observed values,Invalid rows
0,DayOfWeek,"1, 2, 3, 4, 5, 6, 7","1, 2, 3, 4, 5, 6, 7",0
1,Open,"0, 1","0, 1",0
2,Promo,"0, 1","0, 1",0
3,StateHoliday,"0, a, b, c","0, a, b, c",0
4,SchoolHoliday,"0, 1","0, 1",0



STORE categorical-value validation:


,Column,Expected values,Observed values,Invalid rows
0,StoreType,"a, b, c, d","a, b, c, d",0
1,Assortment,"a, b, c","a, b, c",0
2,Promo2,"0, 1","0, 1",0


# 7. Validate Numeric Value Ranges

## Purpose

This section checks whether selected numeric variables contain values outside their logically valid ranges.

The previous categorical validation showed that all inspected categorical and binary variables contain valid categories. However, numeric variables can still contain invalid values even when their data types are correct.

For example:

- `Sales` and `Customers` should not be negative;
- `CompetitionDistance` should not be negative;
- `CompetitionOpenSinceMonth` should be between 1 and 12 when present;
- `Promo2SinceWeek` should be between 1 and 53 when present;
- month, week, and year fields should represent whole numbers rather than fractional values.

This section checks these basic numerical constraints without modifying the data.

## Input

This section uses the DataFrames already loaded into memory:

- `train`
- `store`

No additional files are read.

## Output

No new file is created or modified.

The code produces a validation table showing, for each selected numeric field:

- the minimum observed value;
- the maximum observed value;
- the number of values below the permitted lower bound;
- the number of values above the permitted upper bound, where applicable;
- the number of non-integer values for fields that should represent whole-number units;
- the total number of invalid rows.

Missing values are excluded from this validation because they were investigated separately.

## Method

Explicit numerical validation rules are defined for each selected variable.

Boolean comparisons are used to identify observations that violate a lower or upper bound.

For variables that represent discrete units such as month, week, or year, the code also checks whether the observed values are mathematically equivalent to whole numbers.

For example, a value such as `9.0` is considered a valid whole-number month because it represents the integer 9, whereas `9.5` would be invalid.

This rule-based method is preferred to identifying invalid values using statistical outlier techniques such as z-scores or the interquartile range because the current objective is to detect logically impossible values rather than statistically unusual but potentially valid observations.

Statistical outliers will be investigated separately.

## Principle

A numerical value can be unusual without being invalid.

For example, an unusually high sales value may represent a genuinely busy store and should not automatically be classified as an error.

In contrast, some values violate fixed logical constraints:

- negative customer counts are impossible;
- month 13 is invalid;
- week 60 is invalid.

Therefore, this section performs **domain-range validation** based on known logical boundaries.

For a numeric variable \(X\) with lower bound \(L\) and upper bound \(U\), a valid observation satisfies:

`L ≤ X ≤ U`

when both bounds exist.

For discrete numeric variables, a second condition is applied:

`X = floor(X)`

which ensures that the value represents a whole number.

## Evaluation Criteria

The preferred result is zero invalid rows for every checked field.

The following rules are used:

### `train`

- `Sales` ≥ 0
- `Customers` ≥ 0

### `store`

- `CompetitionDistance` ≥ 0
- `CompetitionOpenSinceMonth`: integer from 1 to 12
- `CompetitionOpenSinceYear`: positive integer
- `Promo2SinceWeek`: integer from 1 to 53
- `Promo2SinceYear`: positive integer

A value that satisfies these rules is considered logically valid at the individual-variable level.

This does not determine whether extreme values are statistical outliers or whether combinations of values across different fields are logically consistent. Those issues are examined separately.

No values are modified in this section.

In [7]:
# Validate selected numeric fields against logical value ranges.

def validate_numeric_range(
    df,
    column,
    lower=None,
    upper=None,
    integer_required=False
):
    """
    Validate the logical numeric range of one column.

    Missing values are excluded because they were inspected separately.
    """

    values = df[column].dropna()

    below_lower = 0
    above_upper = 0
    non_integer = 0

    if lower is not None:
        below_lower = int((values < lower).sum())

    if upper is not None:
        above_upper = int((values > upper).sum())

    if integer_required:
        non_integer = int((values % 1 != 0).sum())

    invalid_mask = pd.Series(False, index=values.index)

    if lower is not None:
        invalid_mask |= values < lower

    if upper is not None:
        invalid_mask |= values > upper

    if integer_required:
        invalid_mask |= values % 1 != 0

    return {
        "Column": column,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Below lower bound": below_lower,
        "Above upper bound": above_upper,
        "Non-integer": non_integer,
        "Invalid rows": int(invalid_mask.sum())
    }


# ------------------------------------------------------------
# 1. Validate numeric fields in train.csv
# ------------------------------------------------------------

train_numeric_rules = [
    {
        "column": "Sales",
        "lower": 0
    },
    {
        "column": "Customers",
        "lower": 0
    }
]

train_numeric_validation = pd.DataFrame([
    validate_numeric_range(train, **rule)
    for rule in train_numeric_rules
])

print("TRAIN numeric-range validation:")
display(train_numeric_validation)


# ------------------------------------------------------------
# 2. Validate numeric fields in store.csv
# ------------------------------------------------------------

store_numeric_rules = [
    {
        "column": "CompetitionDistance",
        "lower": 0
    },
    {
        "column": "CompetitionOpenSinceMonth",
        "lower": 1,
        "upper": 12,
        "integer_required": True
    },
    {
        "column": "CompetitionOpenSinceYear",
        "lower": 1,
        "integer_required": True
    },
    {
        "column": "Promo2SinceWeek",
        "lower": 1,
        "upper": 53,
        "integer_required": True
    },
    {
        "column": "Promo2SinceYear",
        "lower": 1,
        "integer_required": True
    }
]

store_numeric_validation = pd.DataFrame([
    validate_numeric_range(store, **rule)
    for rule in store_numeric_rules
])

print("\nSTORE numeric-range validation:")
display(store_numeric_validation)

TRAIN numeric-range validation:


,Column,Minimum,Maximum,Below lower bound,Above upper bound,Non-integer,Invalid rows
0,Sales,0,41551,0,0,0,0
1,Customers,0,7388,0,0,0,0



STORE numeric-range validation:


,Column,Minimum,Maximum,Below lower bound,Above upper bound,Non-integer,Invalid rows
0,CompetitionDistance,20.0,75860.0,0,0,0,0
1,CompetitionOpenSinceMonth,1.0,12.0,0,0,0,0
2,CompetitionOpenSinceYear,1900.0,2015.0,0,0,0,0
3,Promo2SinceWeek,1.0,50.0,0,0,0,0
4,Promo2SinceYear,2009.0,2015.0,0,0,0,0


# 8. Validate Dates and Day-of-Week Consistency

## Purpose

This section checks whether the `Date` field contains valid calendar dates and whether the recorded `DayOfWeek` values are consistent with the actual calendar.

Although the `Date` column was loaded successfully as a string-based field, successful CSV loading does not guarantee that every value represents a valid date.

The validation therefore examines:

- whether every date can be parsed using the expected `YYYY-MM-DD` format;
- the earliest and latest valid dates;
- whether the dataset contains every calendar date within the observed time range;
- whether `DayOfWeek` agrees with the weekday calculated from `Date`.

No date values are converted permanently in this notebook.

## Input

This section uses the `train` DataFrame already loaded from:

- `data/raw/train.csv`

The relevant fields are:

- `Date`
- `DayOfWeek`

No additional input file is read.

## Output

No file is created or modified.

The code produces:

1. a date-validation summary containing:
   - number of rows;
   - number of unparseable dates;
   - earliest date;
   - latest date;
   - number of unique dates;
   - number of missing calendar dates within the observed range;
   - number of `DayOfWeek` mismatches;

2. example invalid dates, missing calendar dates, or weekday inconsistencies if any are detected.

The parsed dates are stored only in a temporary Series for inspection.

## Method

`pd.to_datetime()` is used with:

- `format="%Y-%m-%d"` to enforce the expected date structure;
- `errors="coerce"` to convert invalid or unparseable values into `NaT` rather than stopping execution.

The number of `NaT` values is then used to identify invalid dates.

`pd.date_range()` is used to generate the complete sequence of calendar dates between the minimum and maximum observed dates. This expected sequence is compared with the unique observed dates to identify any missing days.

For valid dates, Pandas' `.dt.dayofweek` property is used to calculate the actual weekday. Pandas represents Monday as 0 and Sunday as 6, so 1 is added to obtain the 1–7 convention used by the dataset.

This method is preferred to checking only the minimum and maximum string values because minimum and maximum values do not verify that all individual dates are valid or that the weekday field is internally consistent.

## Principle

Date validation involves both **format validity** and **logical consistency**.

A text value may look like a date but still be invalid, for example:

- `2015-13-01`
- `2015-02-30`

Parsing the value using a defined calendar format verifies whether it corresponds to a real date.

A second form of validation compares related variables.

If a record contains a particular date, the weekday associated with that date is mathematically determined by the calendar. Therefore, `Date` and `DayOfWeek` should agree.

This is an example of a **cross-field consistency check**, where two individually valid variables are checked for logical agreement.

## Evaluation Criteria

The preferred result is:

- zero unparseable dates;
- a plausible earliest and latest date;
- zero missing calendar dates within the overall observed date range;
- zero `DayOfWeek` mismatches.

A zero mismatch count indicates that the recorded weekday values are internally consistent with the calendar dates.

Missing calendar dates, if found, do not automatically imply an error. They would require further investigation to determine whether the absence is expected or represents missing observations.

No values are modified in this section.

In [8]:
# Validate the Date field and its consistency with DayOfWeek.

# ------------------------------------------------------------
# 1. Parse dates using the expected format
# ------------------------------------------------------------

parsed_dates = pd.to_datetime(
    train["Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

invalid_date_mask = parsed_dates.isna()
invalid_date_count = int(invalid_date_mask.sum())


# ------------------------------------------------------------
# 2. Inspect the valid date range
# ------------------------------------------------------------

valid_dates = parsed_dates.dropna()

start_date = valid_dates.min()
end_date = valid_dates.max()

unique_dates = pd.DatetimeIndex(
    valid_dates.drop_duplicates().sort_values()
)


# ------------------------------------------------------------
# 3. Check calendar-date coverage
# ------------------------------------------------------------

expected_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

missing_dates = expected_dates.difference(unique_dates)


# ------------------------------------------------------------
# 4. Check DayOfWeek consistency
# ------------------------------------------------------------

# Pandas: Monday = 0, ..., Sunday = 6
# Dataset convention: Monday = 1, ..., Sunday = 7
expected_day_of_week = parsed_dates.dt.dayofweek + 1

weekday_mismatch_mask = (
    parsed_dates.notna()
    & (train["DayOfWeek"] != expected_day_of_week)
)

weekday_mismatch_count = int(
    weekday_mismatch_mask.sum()
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

date_validation = pd.DataFrame({
    "Check": [
        "Total rows",
        "Unparseable dates",
        "Earliest date",
        "Latest date",
        "Unique calendar dates",
        "Missing calendar dates",
        "DayOfWeek mismatches"
    ],
    "Result": [
        f"{len(train):,}",
        f"{invalid_date_count:,}",
        start_date.strftime("%Y-%m-%d"),
        end_date.strftime("%Y-%m-%d"),
        f"{len(unique_dates):,}",
        f"{len(missing_dates):,}",
        f"{weekday_mismatch_count:,}"
    ]
})

display(date_validation)


# ------------------------------------------------------------
# 6. Display examples only when a problem is detected
# ------------------------------------------------------------

if invalid_date_count > 0:
    print("\nExamples of unparseable dates:")
    display(
        train.loc[
            invalid_date_mask,
            ["Store", "Date", "DayOfWeek"]
        ].head(20)
    )


if len(missing_dates) > 0:
    print("\nMissing calendar dates:")
    display(
        pd.DataFrame({
            "Missing Date": missing_dates
        }).head(20)
    )


if weekday_mismatch_count > 0:
    print("\nExamples of DayOfWeek mismatches:")

    weekday_examples = train.loc[
        weekday_mismatch_mask,
        ["Store", "Date", "DayOfWeek"]
    ].copy()

    weekday_examples["ExpectedDayOfWeek"] = (
        expected_day_of_week[weekday_mismatch_mask]
        .astype(int)
        .values
    )

    display(weekday_examples.head(20))

,Check,Result
0,Total rows,"1,017,209"
1,Unparseable dates,0
2,Earliest date,2013-01-01
3,Latest date,2015-07-31
4,Unique calendar dates,942
5,Missing calendar dates,0
6,DayOfWeek mismatches,0


# 9. Check Business Consistency Between Open, Sales, and Customers

## Purpose

This section examines whether the daily store operating status is logically consistent with the recorded sales and customer counts.

The previous date validation found that all dates are parseable, the calendar coverage is complete, and all `DayOfWeek` values agree with the actual calendar.

The current section focuses on three closely related variables:

- `Open`: whether the store was open on that day;
- `Sales`: the recorded daily sales amount;
- `Customers`: the recorded number of customers.

Several cross-field conditions are examined because each variable may contain individually valid values while their combination may still be inconsistent.

## Input

This section uses the `train` DataFrame already loaded from:

- `data/raw/train.csv`

The relevant fields are:

- `Open`
- `Sales`
- `Customers`

No additional input file is read.

## Output

No file is created or modified.

The code produces a business-consistency summary showing the number of rows satisfying several conditions, including:

- closed stores with positive sales;
- closed stores with positive customer counts;
- positive sales with zero customers;
- positive customers with zero sales;
- open stores with zero sales;
- open stores with zero customers.

Example records are displayed only when a potentially inconsistent condition is detected.

## Method

Boolean conditions are constructed using direct comparisons between the three variables.

For example:

- `Open == 0` identifies closed-store records;
- `Sales > 0` identifies records with positive sales;
- combining both conditions with the Boolean AND operator (`&`) identifies records where a store is recorded as closed but still has positive sales.

This rule-based cross-field validation is preferred to correlation analysis or statistical anomaly detection because the objective is to test logical relationships between variables rather than statistical association.

The conditions are separated into two categories:

1. **strong consistency checks**, where the combination would normally contradict the meaning of `Open`;
2. **review conditions**, where the pattern is unusual but may still have a legitimate business explanation.

No records are classified as errors solely because they satisfy a review condition.

## Principle

Cross-field consistency checks evaluate whether related variables agree with each other.

A record may pass all individual field checks while still containing a logical contradiction.

For example:

- `Open = 0`
- `Sales = 5000`

Both values are individually valid:

- `Open = 0` is an allowed binary value;
- `Sales = 5000` is a valid non-negative number.

However, their combination is potentially inconsistent because a store recorded as closed would normally not generate daily in-store sales.

Similarly, combinations involving zero sales and zero customers require careful interpretation. An open store may theoretically record no transactions on a particular day, so such cases should be investigated rather than automatically treated as errors.

## Evaluation Criteria

The strongest consistency expectations are:

- when `Open = 0`, `Sales` should normally be 0;
- when `Open = 0`, `Customers` should normally be 0.

Therefore, the preferred result is zero records for:

- closed stores with positive sales;
- closed stores with positive customers.

The following conditions are treated as review cases rather than automatic errors:

- positive sales with zero customers;
- positive customers with zero sales;
- open stores with zero sales;
- open stores with zero customers.

If these cases occur, their frequency and overlap should be examined before deciding whether any cleaning is required.

No values are modified in this section.

In [9]:
# Check logical consistency between store opening status, sales, and customers.

# ------------------------------------------------------------
# 1. Define cross-field conditions
# ------------------------------------------------------------

closed_with_sales = (
    (train["Open"] == 0)
    & (train["Sales"] > 0)
)

closed_with_customers = (
    (train["Open"] == 0)
    & (train["Customers"] > 0)
)

sales_without_customers = (
    (train["Sales"] > 0)
    & (train["Customers"] == 0)
)

customers_without_sales = (
    (train["Customers"] > 0)
    & (train["Sales"] == 0)
)

open_with_zero_sales = (
    (train["Open"] == 1)
    & (train["Sales"] == 0)
)

open_with_zero_customers = (
    (train["Open"] == 1)
    & (train["Customers"] == 0)
)


# ------------------------------------------------------------
# 2. Build a summary table
# ------------------------------------------------------------

business_consistency = pd.DataFrame({
    "Condition": [
        "Closed store with positive sales",
        "Closed store with positive customers",
        "Positive sales with zero customers",
        "Positive customers with zero sales",
        "Open store with zero sales",
        "Open store with zero customers"
    ],
    "Count": [
        int(closed_with_sales.sum()),
        int(closed_with_customers.sum()),
        int(sales_without_customers.sum()),
        int(customers_without_sales.sum()),
        int(open_with_zero_sales.sum()),
        int(open_with_zero_customers.sum())
    ],
    "Assessment": [
        "Strong inconsistency",
        "Strong inconsistency",
        "Review",
        "Review",
        "Review",
        "Review"
    ]
})

display(business_consistency)


# ------------------------------------------------------------
# 3. Display examples only when relevant cases are found
# ------------------------------------------------------------

strong_inconsistency_mask = (
    closed_with_sales
    | closed_with_customers
)

if strong_inconsistency_mask.any():
    print("\nExamples of strong business inconsistencies:")
    display(
        train.loc[
            strong_inconsistency_mask,
            [
                "Store",
                "Date",
                "Open",
                "Sales",
                "Customers"
            ]
        ].head(20)
    )


review_mask = (
    sales_without_customers
    | customers_without_sales
    | open_with_zero_sales
    | open_with_zero_customers
)

if review_mask.any():
    print("\nExamples of records requiring review:")
    display(
        train.loc[
            review_mask,
            [
                "Store",
                "Date",
                "Open",
                "Sales",
                "Customers"
            ]
        ].head(20)
    )

,Condition,Count,Assessment
0,Closed store with positive sales,0,Strong inconsistency
1,Closed store with positive customers,0,Strong inconsistency
2,Positive sales with zero customers,0,Review
3,Positive customers with zero sales,2,Review
4,Open store with zero sales,54,Review
5,Open store with zero customers,52,Review



Examples of records requiring review:


,Store,Date,Open,Sales,Customers
86825,971,2015-05-15,1,0,0
142278,674,2015-03-26,1,0,0
196938,699,2015-02-05,1,0,0
322053,708,2014-10-01,1,0,0
330176,357,2014-09-22,1,0,0
340348,227,2014-09-11,1,0,0
340860,835,2014-09-11,1,0,0
341795,835,2014-09-10,1,0,0
346232,548,2014-09-05,1,0,0
346734,28,2014-09-04,1,0,0


# 10. Review Open-Store Records with Zero Sales

## Observation

The previous business-consistency check found no records where a closed store had positive sales or positive customer counts.

However, 54 records were identified where the store was marked as open but recorded zero sales. Most of these records also had zero customers, while a small number recorded positive customer counts.

These cases are reviewed further before deciding whether they represent errors or legitimate operational situations.

## Purpose

This section investigates the structure and distribution of open-store records with zero sales.

The analysis determines:

- how many records also have zero customers;
- how many records have positive customers despite zero sales;
- how many stores are affected;
- whether the cases are concentrated in particular stores or dates.

The objective is to understand the pattern rather than automatically classify these records as errors.

## Input

This section uses the `train` DataFrame already loaded from:

- `data/raw/train.csv`

The relevant fields are:

- `Store`
- `Date`
- `Open`
- `Sales`
- `Customers`

No additional file is read.

## Output

No file is created or modified.

The code displays:

1. a summary of the zero-sales patterns;
2. the stores most frequently affected;
3. the dates with the largest number of affected stores;
4. all records where positive customers were recorded despite zero sales.

## Method

The analysis first isolates records satisfying:

`Open = 1` and `Sales = 0`

These observations are then separated according to whether `Customers` is zero or positive.

`groupby()` is used to count the number of affected records by store and by date.

This method is preferred to immediately removing zero-sales records because zero sales can be a legitimate observation. The pattern must first be examined to determine whether it reflects data errors, unusual business activity, or valid operating conditions.

## Principle

A statistically unusual record is not necessarily an invalid record.

An open store can theoretically record zero sales. Therefore, the combination:

`Open = 1, Sales = 0`

is treated as a review condition rather than a confirmed error.

However, the combination:

`Customers > 0, Sales = 0`

requires closer examination because it indicates that customers were recorded while no sales were recorded.

Examining whether such cases are isolated or systematically concentrated in particular stores or dates can provide evidence about their likely cause.

## Evaluation Criteria

The results should be evaluated by considering:

1. whether most zero-sales observations also have zero customers;
2. whether the same stores repeatedly experience the condition;
3. whether many stores experience the condition on the same dates;
4. whether the two records with positive customers and zero sales contain any obvious pattern.

These observations will be recorded for later cleaning decisions.

No rows are removed or modified in this section.

In [10]:
# Investigate open-store records with zero sales.

# ------------------------------------------------------------
# 1. Isolate open stores with zero sales
# ------------------------------------------------------------

open_zero_sales = train.loc[
    (train["Open"] == 1)
    & (train["Sales"] == 0)
].copy()


# ------------------------------------------------------------
# 2. Separate the two customer-count patterns
# ------------------------------------------------------------

zero_customers = (
    open_zero_sales["Customers"] == 0
)

positive_customers = (
    open_zero_sales["Customers"] > 0
)


zero_sales_summary = pd.DataFrame({
    "Condition": [
        "Open with zero sales",
        "Open with zero sales and zero customers",
        "Open with zero sales and positive customers"
    ],
    "Count": [
        len(open_zero_sales),
        int(zero_customers.sum()),
        int(positive_customers.sum())
    ]
})

display(zero_sales_summary)


# ------------------------------------------------------------
# 3. Check concentration by store
# ------------------------------------------------------------

zero_sales_by_store = (
    open_zero_sales
    .groupby("Store")
    .size()
    .reset_index(name="Zero-sales records")
    .sort_values(
        "Zero-sales records",
        ascending=False
    )
)

print("\nStores with the most open zero-sales records:")
display(zero_sales_by_store.head(10))


# ------------------------------------------------------------
# 4. Check concentration by date
# ------------------------------------------------------------

zero_sales_by_date = (
    open_zero_sales
    .groupby("Date")
    .size()
    .reset_index(name="Affected stores")
    .sort_values(
        ["Affected stores", "Date"],
        ascending=[False, True]
    )
)

print("\nDates with the most open zero-sales stores:")
display(zero_sales_by_date.head(10))


# ------------------------------------------------------------
# 5. Inspect the strongest review cases
# ------------------------------------------------------------

positive_customer_zero_sales = open_zero_sales.loc[
    positive_customers,
    [
        "Store",
        "Date",
        "DayOfWeek",
        "Sales",
        "Customers",
        "Promo",
        "StateHoliday",
        "SchoolHoliday"
    ]
]

print("\nOpen stores with positive customers but zero sales:")
display(positive_customer_zero_sales)

,Condition,Count
0,Open with zero sales,54
1,Open with zero sales and zero customers,52
2,Open with zero sales and positive customers,2



Stores with the most open zero-sales records:


,Store,Zero-sales records
1,28,3
0,25,2
3,102,2
40,1100,2
36,983,2
18,623,2
21,665,2
13,364,2
10,339,2
38,1017,2



Dates with the most open zero-sales stores:


,Date,Affected stores
37,2014-07-24,4
36,2014-07-23,2
43,2014-09-11,2
0,2013-01-17,1
1,2013-01-24,1
2,2013-01-30,1
3,2013-01-31,1
4,2013-02-07,1
5,2013-03-16,1
6,2013-04-25,1



Open stores with positive customers but zero sales:


,Store,Date,DayOfWeek,Sales,Customers,Promo,StateHoliday,SchoolHoliday
478649,1100,2014-04-29,2,0,3,1,0,0
889932,948,2013-04-25,4,0,5,1,0,0


# 11. Check Store ID Consistency Across Datasets

## Observation

The previous business-consistency review identified 54 open-store records with zero sales. Of these, 52 also had zero customers and only 2 had positive customer counts.

The cases were distributed across multiple stores and dates, with no strong concentration indicating an obvious systematic failure. They are therefore retained as review cases rather than confirmed data errors.

## Purpose

This section checks whether the store identifiers are consistent between `train.csv` and `store.csv`.

The daily sales dataset uses `Store` to identify each store, while the store information dataset uses the same field to provide store-level attributes.

Before the datasets are combined later in the project, it is necessary to verify that:

- every store appearing in `train` has a corresponding record in `store`;
- every store in `store` appears in the historical training data;
- both datasets cover the same set of store identifiers.

This is a referential-integrity check and does not modify either dataset.

## Input

This section uses:

- `train`, originating from `data/raw/train.csv`
- `store`, originating from `data/raw/store.csv`

The relevant field in both datasets is:

- `Store`

No additional file is read.

## Output

No file is created or modified.

The code produces a summary containing:

- the number of unique stores in `train`;
- the number of unique stores in `store`;
- the number of store IDs present in `train` but missing from `store`;
- the number of store IDs present in `store` but missing from `train`;
- whether both datasets contain exactly the same set of store IDs.

If unmatched store IDs are detected, they are displayed for further investigation.

## Method

The unique values of `Store` are converted into Python sets.

Set difference is then used to identify unmatched identifiers:

- `train_stores - store_stores` finds stores used in the daily sales data but absent from the store information;
- `store_stores - train_stores` finds store-information records that do not appear in the daily sales data.

Set equality is used to determine whether the two datasets contain exactly the same store identifiers.

This method is preferred to performing a full DataFrame merge at this stage because only key coverage is being tested. Set operations provide a direct and efficient way to compare unique identifiers without creating a larger temporary joined dataset.

## Principle

When two tables are linked through a common identifier, the identifier relationship should be validated before the tables are merged.

This property is commonly referred to as **referential integrity**.

For this project, each `Store` value in the daily sales table should refer to a valid store record in the store-information table.

If:

`Store(train) ⊆ Store(store)`

then every daily record has a corresponding store-information record.

If both sets are equal:

`Store(train) = Store(store)`

then the two datasets have complete and identical store coverage.

Failure of this condition could cause unmatched records or missing store attributes during later data integration.

## Evaluation Criteria

The preferred result is:

- the same number of unique stores in both datasets;
- zero store IDs present only in `train`;
- zero store IDs present only in `store`;
- exact equality between the two sets of store IDs.

If unmatched store IDs are detected, the affected stores must be investigated before the datasets are merged.

No rows or identifiers are modified in this section.

In [11]:
# Check referential integrity of Store identifiers across train.csv and store.csv.

# ------------------------------------------------------------
# 1. Extract unique Store identifiers
# ------------------------------------------------------------

train_stores = set(train["Store"].dropna().unique())
store_stores = set(store["Store"].dropna().unique())


# ------------------------------------------------------------
# 2. Find unmatched Store identifiers
# ------------------------------------------------------------

only_in_train = sorted(
    train_stores - store_stores
)

only_in_store = sorted(
    store_stores - train_stores
)


# ------------------------------------------------------------
# 3. Check whether both datasets contain the same Store IDs
# ------------------------------------------------------------

same_store_ids = (
    train_stores == store_stores
)


# ------------------------------------------------------------
# 4. Build summary
# ------------------------------------------------------------

store_id_validation = pd.DataFrame({
    "Check": [
        "Unique stores in train",
        "Unique stores in store",
        "Stores only in train",
        "Stores only in store",
        "Identical Store ID sets"
    ],
    "Result": [
        f"{len(train_stores):,}",
        f"{len(store_stores):,}",
        f"{len(only_in_train):,}",
        f"{len(only_in_store):,}",
        str(same_store_ids)
    ]
})

display(store_id_validation)


# ------------------------------------------------------------
# 5. Display unmatched IDs only if they exist
# ------------------------------------------------------------

if only_in_train:
    print("\nStore IDs present in train but missing from store:")
    print(only_in_train)

if only_in_store:
    print("\nStore IDs present in store but missing from train:")
    print(only_in_store)

,Check,Result
0,Unique stores in train,"1,115"
1,Unique stores in store,"1,115"
2,Stores only in train,0
3,Stores only in store,0
4,Identical Store ID sets,True


# 12. Inspect Potential Numerical Outliers

## Observation

The store-identifier consistency check confirmed that `train` and `store` contain exactly the same 1,115 store IDs, with no unmatched stores in either dataset.

The current section moves from logical validity checks to statistical outlier inspection.

## Purpose

This section identifies unusually high or low observations in selected continuous numeric variables.

The variables examined are:

- `Sales`
- `Customers`
- `CompetitionDistance`

These variables are selected because they represent continuous or count-based quantities for which unusually extreme observations may be meaningful.

Identifier variables, binary variables, categorical codes, calendar fields, and year/week/month variables are not treated as continuous outlier-analysis variables.

For `Sales` and `Customers`, only records with `Open = 1` are used when calculating the statistical thresholds. Closed-store records contain structural zeros and should not influence the normal operating distribution.

## Input

This section uses:

- `train`, originating from `data/raw/train.csv`
- `store`, originating from `data/raw/store.csv`

No additional file is read.

## Output

No file is created or modified.

The code produces:

1. an IQR-based outlier summary for each selected variable;
2. the first quartile (`Q1`);
3. the third quartile (`Q3`);
4. the interquartile range (`IQR`);
5. the lower and upper outlier boundaries;
6. the number and percentage of observations outside these boundaries;
7. several of the most extreme high-value observations for manual inspection.

The results identify potential statistical outliers only. They do not classify these observations as data errors.

## Method

The Interquartile Range (IQR) method is used.

For a numeric variable:

`IQR = Q3 - Q1`

where:

- `Q1` is the 25th percentile;
- `Q3` is the 75th percentile.

The conventional outlier boundaries are:

`Lower Bound = Q1 - 1.5 × IQR`

`Upper Bound = Q3 + 1.5 × IQR`

Observations outside these boundaries are flagged as potential outliers.

For `Sales` and `Customers`, the thresholds are calculated only from records where `Open = 1`, because zeros associated with closed stores are structurally generated by the operating status rather than by the normal distribution of active-store activity.

The IQR method is preferred to the z-score method because retail sales, customer counts, and competition distance can be strongly skewed. The IQR method is based on quartiles and is therefore less sensitive to extreme observations than statistics based on the mean and standard deviation.

## Principle

An outlier is an observation that lies unusually far from the central portion of a distribution.

The IQR method uses the middle 50% of observations as a robust description of the typical range.

Because quartiles are based on order rather than the magnitude of every observation, an extremely large value has relatively little influence on `Q1` and `Q3`.

This makes the method suitable for skewed business data.

However, an IQR outlier is not automatically an invalid observation.

For example, unusually high sales may result from:

- a particularly large store;
- a promotional event;
- seasonal demand;
- unusually high customer traffic.

Similarly, a large `CompetitionDistance` may genuinely indicate that the nearest competitor is far away.

Therefore, outlier detection is used here as a screening method rather than a deletion rule.

## Evaluation Criteria

The results should be evaluated by considering:

1. the number and percentage of observations flagged as potential outliers;
2. the magnitude of the most extreme observations;
3. whether the values remain logically possible;
4. whether the extreme observations can be explained by other variables or business conditions.

A non-zero outlier count does not imply poor data quality.

Only observations that are both statistically unusual and supported by additional evidence of error should later be considered for correction or removal.

No observations are removed or modified in this section.

In [12]:
# Identify potential statistical outliers using the IQR method.

def iqr_outlier_summary(series, variable_name):
    """
    Calculate IQR-based outlier thresholds and summary statistics.
    
    Missing values are excluded automatically.
    """

    values = series.dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (values < lower_bound)
        | (values > upper_bound)
    )

    outlier_count = int(outlier_mask.sum())
    outlier_percentage = (
        outlier_count / len(values) * 100
    )

    return {
        "Variable": variable_name,
        "Observations": len(values),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower bound": lower_bound,
        "Upper bound": upper_bound,
        "Outliers": outlier_count,
        "Outliers (%)": round(outlier_percentage, 2)
    }


# ------------------------------------------------------------
# 1. Use operating-store records for Sales and Customers
# ------------------------------------------------------------

open_records = train.loc[
    train["Open"] == 1
]


# ------------------------------------------------------------
# 2. Calculate IQR summaries
# ------------------------------------------------------------

outlier_summary = pd.DataFrame([
    iqr_outlier_summary(
        open_records["Sales"],
        "Sales (open stores)"
    ),
    iqr_outlier_summary(
        open_records["Customers"],
        "Customers (open stores)"
    ),
    iqr_outlier_summary(
        store["CompetitionDistance"],
        "CompetitionDistance"
    )
])

display(outlier_summary)


# ------------------------------------------------------------
# 3. Inspect the largest Sales observations
# ------------------------------------------------------------

print("\nHighest Sales observations:")

display(
    train.loc[
        train["Open"] == 1,
        [
            "Store",
            "Date",
            "Sales",
            "Customers",
            "Promo",
            "StateHoliday",
            "SchoolHoliday"
        ]
    ]
    .sort_values("Sales", ascending=False)
    .head(10)
)


# ------------------------------------------------------------
# 4. Inspect the largest Customers observations
# ------------------------------------------------------------

print("\nHighest Customers observations:")

display(
    train.loc[
        train["Open"] == 1,
        [
            "Store",
            "Date",
            "Sales",
            "Customers",
            "Promo",
            "StateHoliday",
            "SchoolHoliday"
        ]
    ]
    .sort_values("Customers", ascending=False)
    .head(10)
)


# ------------------------------------------------------------
# 5. Inspect the largest CompetitionDistance observations
# ------------------------------------------------------------

print("\nLargest CompetitionDistance observations:")

display(
    store[
        [
            "Store",
            "StoreType",
            "Assortment",
            "CompetitionDistance"
        ]
    ]
    .dropna(subset=["CompetitionDistance"])
    .sort_values(
        "CompetitionDistance",
        ascending=False
    )
    .head(10)
)

,Variable,Observations,Q1,Q3,IQR,Lower bound,Upper bound,Outliers,Outliers (%)
0,Sales (open stores),844392,4859.0,8360.0,3501.0,-392.5,13611.5,30769,3.64
1,Customers (open stores),844392,519.0,893.0,374.0,-42.0,1454.0,40853,4.84
2,CompetitionDistance,1112,717.5,6882.5,6165.0,-8530.0,16130.0,108,9.71



Highest Sales observations:


,Store,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
44393,909,2015-06-22,41551,1721,0,0,0
132946,262,2015-04-03,38722,5132,1,b,0
101726,262,2015-05-01,38484,5458,1,a,0
87231,262,2015-05-14,38367,5192,0,a,1
424086,57,2014-06-16,38037,1970,1,0,0
627776,817,2013-12-16,38025,4381,1,0,0
627220,261,2013-12-16,37646,1964,1,0,0
444361,262,2014-05-29,37403,5297,0,a,0
620531,262,2013-12-22,37376,4916,0,0,0
245945,262,2014-12-21,37122,4962,0,0,0



Highest Customers observations:


,Store,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
993496,817,2013-01-22,27190,7388,1,0,0
319810,262,2014-10-03,35702,5494,1,a,0
101726,262,2015-05-01,38484,5458,1,a,0
432096,262,2014-06-09,34692,5387,0,a,0
444361,262,2014-05-29,37403,5297,0,a,0
87231,262,2015-05-14,38367,5192,0,a,1
129601,262,2015-04-06,33655,5152,0,b,0
745411,262,2013-09-01,32926,5145,0,0,0
132946,262,2015-04-03,38722,5132,1,b,0
776631,262,2013-08-04,32252,5112,0,0,0



Largest CompetitionDistance observations:


,Store,StoreType,Assortment,CompetitionDistance
452,453,a,c,75860.0
121,122,a,c,58260.0
800,801,d,a,48330.0
109,110,a,c,46590.0
746,747,c,c,45740.0
461,462,a,a,44320.0
523,524,a,c,40860.0
725,726,a,c,40540.0
314,315,a,c,38710.0
298,299,d,c,38630.0


# 13. Consolidate Data Quality Findings

## Purpose

This section consolidates the results of all previous data-quality checks into a single structured findings table.

No new data-quality test is introduced here. Instead, the previously observed issues and validation results are classified according to their meaning and the action required in the next stage of the project.

The table distinguishes between:

- confirmed data issues;
- structural missingness;
- missing information;
- review cases;
- statistical outliers;
- checks where no issue was detected.

This separation is important because different types of findings should not automatically receive the same cleaning treatment.

## Input

This section uses the `train` and `store` DataFrames already loaded from:

- `data/raw/train.csv`
- `data/raw/store.csv`

It also uses results established in the previous inspection sections.

No additional input file is read.

## Output

No file is created or modified.

The code produces one DataFrame named `quality_findings`.

The table contains four fields:

- `Finding`: the data-quality item being assessed;
- `Evidence`: the observed result from the inspection;
- `Classification`: the interpretation of the result;
- `Next action`: the recommended treatment in the next project stage.

This table provides the transition from `01_check.ipynb` to `02_clean.ipynb`.

## Method

The findings are constructed from the results already established in the previous sections.

The table does not apply additional transformations or statistical tests. Instead, it applies a classification framework to distinguish different kinds of data-quality findings.

For example:

- inconsistent representations of the same category are classified as a confirmed data issue;
- missing values caused by a variable being not applicable are classified as structural missingness;
- logically valid but unusual records are classified as review cases;
- observations identified only by the IQR rule are classified as statistical outliers rather than errors.

This approach is preferred to creating a single generic "error count" because data-quality problems differ in cause, severity, and appropriate treatment.

## Principle

Data cleaning should be based on the meaning of a detected problem rather than on the presence of an unusual value alone.

A useful distinction is:

**Detection → Interpretation → Action**

For example:

- detecting `NaN` does not determine whether it should be filled;
- detecting an IQR outlier does not determine whether it should be removed;
- detecting two representations of the same category provides stronger evidence that normalization is required.

Therefore, the findings are classified before any cleaning operation is performed.

## Evaluation Criteria

A finding is classified as a **confirmed issue** when the data representation or information is demonstrably inconsistent or incomplete.

A finding is classified as **structural missingness** when the missing value is explained by the logical structure of another variable.

A **review case** is unusual but not sufficiently supported as an error.

A **statistical outlier** is unusual according to a statistical rule but remains logically possible.

A **no issue detected** result means that the corresponding validation rule was satisfied by all inspected records.

Only findings requiring an explicit data transformation should become mandatory cleaning actions in `02_clean.ipynb`.

In [13]:
# Consolidate all confirmed findings and review cases from the inspection stage.

# ------------------------------------------------------------
# 1. Extract selected results from previous inspections
# ------------------------------------------------------------

promo2_structural_missing = int(
    (
        (store["Promo2"] == 0)
        & store[
            [
                "Promo2SinceWeek",
                "Promo2SinceYear",
                "PromoInterval"
            ]
        ].isna().all(axis=1)
    ).sum()
)

competition_date_missing = int(
    (
        store["CompetitionOpenSinceMonth"].isna()
        & store["CompetitionOpenSinceYear"].isna()
    ).sum()
)

competition_distance_missing = int(
    store["CompetitionDistance"].isna().sum()
)

open_zero_sales_count = int(
    (
        (train["Open"] == 1)
        & (train["Sales"] == 0)
    ).sum()
)

positive_customers_zero_sales_count = int(
    (
        (train["Open"] == 1)
        & (train["Sales"] == 0)
        & (train["Customers"] > 0)
    ).sum()
)


# Extract IQR outlier percentages from the previous summary.
sales_outlier_pct = float(
    outlier_summary.loc[
        outlier_summary["Variable"] == "Sales (open stores)",
        "Outliers (%)"
    ].iloc[0]
)

customers_outlier_pct = float(
    outlier_summary.loc[
        outlier_summary["Variable"] == "Customers (open stores)",
        "Outliers (%)"
    ].iloc[0]
)

competition_outlier_pct = float(
    outlier_summary.loc[
        outlier_summary["Variable"] == "CompetitionDistance",
        "Outliers (%)"
    ].iloc[0]
)


# ------------------------------------------------------------
# 2. Build the consolidated findings table
# ------------------------------------------------------------

quality_findings = pd.DataFrame([
    {
        "Finding": "StateHoliday representation",
        "Evidence": (
            "'0' stored as str in 855,087 rows and "
            "0 stored as int in 131,072 rows"
        ),
        "Classification": "Confirmed issue",
        "Next action": "Normalize to one consistent categorical type"
    },
    {
        "Finding": "Promo2 detail missing values",
        "Evidence": (
            f"{promo2_structural_missing} stores with Promo2=0 "
            "have all Promo2 detail fields missing"
        ),
        "Classification": "Structural missingness",
        "Next action": "Preserve not-applicable meaning; do not use statistical imputation"
    },
    {
        "Finding": "Competition opening date missing",
        "Evidence": (
            f"{competition_date_missing} stores have both competition "
            "opening month and year missing"
        ),
        "Classification": "Missing information",
        "Next action": "Handle explicitly without assuming a false date"
    },
    {
        "Finding": "CompetitionDistance missing",
        "Evidence": (
            f"{competition_distance_missing} stores have missing distance"
        ),
        "Classification": "Missing information",
        "Next action": "Handle explicitly during cleaning or preprocessing"
    },
    {
        "Finding": "Duplicate records",
        "Evidence": "No full-row or business-key duplicates detected",
        "Classification": "No issue detected",
        "Next action": "No duplicate removal required"
    },
    {
        "Finding": "Categorical value domains",
        "Evidence": "All checked categorical and binary values are valid",
        "Classification": "No issue detected",
        "Next action": "No correction required"
    },
    {
        "Finding": "Numeric value ranges",
        "Evidence": "All checked numeric values satisfy logical range rules",
        "Classification": "No issue detected",
        "Next action": "No range correction required"
    },
    {
        "Finding": "Date consistency",
        "Evidence": (
            "0 unparseable dates, 0 missing calendar dates, "
            "0 DayOfWeek mismatches"
        ),
        "Classification": "No issue detected",
        "Next action": "Convert Date to datetime during cleaning"
    },
    {
        "Finding": "Store ID consistency",
        "Evidence": (
            "1,115 stores in each dataset with identical Store ID sets"
        ),
        "Classification": "No issue detected",
        "Next action": "Store can be used safely as the merge key"
    },
    {
        "Finding": "Open-store zero-sales records",
        "Evidence": (
            f"{open_zero_sales_count} records; "
            f"{positive_customers_zero_sales_count} also have positive customers"
        ),
        "Classification": "Review case",
        "Next action": "Retain unless further evidence indicates an error"
    },
    {
        "Finding": "IQR potential outliers",
        "Evidence": (
            f"Sales {sales_outlier_pct:.2f}%, "
            f"Customers {customers_outlier_pct:.2f}%, "
            f"CompetitionDistance {competition_outlier_pct:.2f}%"
        ),
        "Classification": "Statistical outliers",
        "Next action": "Retain; investigate further during EDA and modelling"
    }
])


# ------------------------------------------------------------
# 3. Display final inspection findings
# ------------------------------------------------------------

display(quality_findings)

,Finding,Evidence,Classification,Next action
0,StateHoliday representation,"'0' stored as str in 855,087 rows and 0 stored...",Confirmed issue,Normalize to one consistent categorical type
1,Promo2 detail missing values,544 stores with Promo2=0 have all Promo2 detai...,Structural missingness,Preserve not-applicable meaning; do not use st...
2,Competition opening date missing,354 stores have both competition opening month...,Missing information,Handle explicitly without assuming a false date
3,CompetitionDistance missing,3 stores have missing distance,Missing information,Handle explicitly during cleaning or preproces...
4,Duplicate records,No full-row or business-key duplicates detected,No issue detected,No duplicate removal required
5,Categorical value domains,All checked categorical and binary values are ...,No issue detected,No correction required
6,Numeric value ranges,All checked numeric values satisfy logical ran...,No issue detected,No range correction required
7,Date consistency,"0 unparseable dates, 0 missing calendar dates,...",No issue detected,Convert Date to datetime during cleaning
8,Store ID consistency,"1,115 stores in each dataset with identical St...",No issue detected,Store can be used safely as the merge key
9,Open-store zero-sales records,54 records; 2 also have positive customers,Review case,Retain unless further evidence indicates an error


# 14. Summary of Data Quality Findings

The raw Rossmann Store Sales datasets were systematically inspected before any cleaning or transformation was performed.

## Dataset Structure and Integrity

The main training dataset contains **1,017,209 daily observations and 9 variables**, while the store information dataset contains **1,115 store records and 10 variables**.

Both datasets were loaded successfully and were non-empty.

No full-row duplicates were detected in either dataset.

The expected business keys were also unique:

- each `Store`-`Date` combination in `train` occurs only once;
- each `Store` value in `store` occurs only once.

Both datasets contain exactly **1,115 unique store IDs**, with no unmatched stores in either direction. Therefore, `Store` can be used as the key for later integration of the two datasets.

## Missing Values

No missing values were found in `train`.

Several variables in `store` contain missing values.

The three `Promo2` detail variables:

- `Promo2SinceWeek`
- `Promo2SinceYear`
- `PromoInterval`

are missing for exactly **544 stores**, and all 544 of these stores have `Promo2 = 0`.

All **571 stores with `Promo2 = 1`** contain complete values for all three detail fields.

This pattern indicates **structural missingness** rather than accidental data loss. The missing values represent information that is not applicable to stores that do not participate in `Promo2`.

Competition-related variables show a different missing-data pattern.

`CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` are both missing for **354 stores**. These two variables are always missing together.

Among these stores, **351 still contain a valid `CompetitionDistance`**, indicating that the competitor is known but its opening date is unavailable.

`CompetitionDistance` itself is missing for only **3 stores**.

These competition-related missing values therefore represent unavailable information and require explicit handling during preprocessing rather than automatic statistical imputation.

## Data Type Consistency

A data representation inconsistency was identified in `StateHoliday`.

The no-holiday category is represented in two different Python types:

- string `"0"` in **855,087 records**;
- integer `0` in **131,072 records**.

The remaining categories `a`, `b`, and `c` are stored as strings.

Although all observed `StateHoliday` categories are semantically valid, the inconsistent representation of the `0` category should be normalized to a single categorical type during data cleaning.

The `Date` field was initially loaded as a string. This is not a data error, but it should be converted to a datetime type during the cleaning stage after successful date validation.

## Categorical and Numeric Validity

All inspected categorical and binary variables contain only valid values.

The following variables showed no invalid categories:

- `DayOfWeek`
- `Open`
- `Promo`
- `StateHoliday`
- `SchoolHoliday`
- `StoreType`
- `Assortment`
- `Promo2`

All inspected numeric variables also satisfied their basic logical range constraints.

No negative values were found in:

- `Sales`
- `Customers`
- `CompetitionDistance`

The month, week, and year variables were within their expected ranges and contained only whole-number values when present.

## Date Validation

All **1,017,209 date values** were successfully parsed using the expected `YYYY-MM-DD` format.

The dataset covers the period from:

**2013-01-01 to 2015-07-31**

This corresponds to **942 unique consecutive calendar dates**.

No calendar dates were missing within this period.

The recorded `DayOfWeek` values were also compared with the weekdays calculated directly from the calendar dates, and **no mismatches were detected**.

The date information is therefore internally consistent.

## Business-Rule Consistency

No records were found where a store was marked as closed (`Open = 0`) while recording positive sales or positive customer counts.

A small number of unusual but logically possible observations were identified among open stores:

- **54 records** have `Open = 1` and `Sales = 0`;
- **52 of these records** also have `Customers = 0`;
- **2 records** contain positive customer counts despite zero sales.

These cases are distributed across multiple stores and dates and do not show strong evidence of a systematic data error.

They are therefore retained as **review cases** rather than classified as invalid records.

## Potential Statistical Outliers

Potential outliers were screened using the **1.5 × IQR rule**.

For operating-store observations:

- **3.64%** of `Sales` observations were flagged as potential IQR outliers;
- **4.84%** of `Customers` observations were flagged as potential IQR outliers.

For `CompetitionDistance`:

- **9.71%** of non-missing observations were flagged as potential IQR outliers.

Inspection of the most extreme observations showed that high sales values frequently coincide with high customer counts, and the extreme values remain within logically possible ranges.

The IQR results therefore indicate statistical extremity rather than confirmed data errors.

These observations should not be removed automatically and will be retained for later exploratory analysis and modelling.

## Issues Requiring Action in the Cleaning Stage

The inspection identified the following items that require attention in `02_clean.ipynb`:

1. Normalize the inconsistent `StateHoliday` representations to one categorical type.
2. Convert `Date` from string to datetime.
3. Preserve the structural meaning of missing `Promo2` detail fields rather than applying ordinary statistical imputation.
4. Handle missing competition information explicitly without creating artificial dates or distances.
5. Retain the rare zero-sales review cases unless additional evidence later indicates that they are erroneous.
6. Retain statistically identified IQR outliers unless later analysis provides evidence that individual observations are invalid.

Overall, the raw datasets show **high structural consistency and generally good data quality**. The main cleaning requirements involve data-type normalization and appropriate treatment of missing information rather than extensive correction or record removal.